# Feature selection — iBABC-CGO (island Binary ABC + Chaos Game Optimization)

Wrapper method theo notebook `fair-comparison/ibabccgo.ipynb` (Nssibi et al., 2024; bản gốc cũng 30 ong × 100 vòng): 5 đảo, di cư CGO mỗi 5 vòng (20% mỗi đảo, chọn hạt giống tốt nhất trong 4), scout limit = 0.1·SN·d, transfer function S, reset 20% quần thể sau 10 vòng không cải thiện. **Khác bản gốc:** (1) nhị phân hoá `rand < tf(x)` → chọn `N_FEATURES` gene có `tf(x) − rand` lớn nhất; (2) fitness cực đại hoá nên xác suất onlooker/roulette di cư đảo chiều tương ứng; (3) trả về đúng tập gene đã được đánh giá tốt nhất (bản gốc giải mã ngẫu nhiên lại global best ở cuối). **Quần thể / số thế hệ thống nhất cho 5 wrapper:** 30 cá thể × 100 thế hệ (vòng lặp). **Fitness dùng chung:** balanced accuracy của `SVC(kernel="linear", class_weight="balanced")`, train trên 80% train của fold và đánh giá trên 20% holdout còn lại (tách một lần, stratified; không dùng val/test). Vì số gene đã cố định nên fitness không có hạng phạt số gene. **Hiệu chỉnh C:** C = 1e-4 tune trên toàn bộ ~12.5k gene quá mạnh khi chỉ còn `N_FEATURES` gene (fitness phẳng 0.5), nên mặc định `C = "auto"`: mỗi fold chọn C trên lưới theo balanced accuracy trung bình của vài tập `N_FEATURES` gene ngẫu nhiên (chỉ dùng train; C đã chọn ghi ở cột `fitness_C`). **Ràng buộc |S| = `N_FEATURES`:** mọi lời giải được giải mã thành đúng `N_FEATURES` gene (xem docstring). Cột `rank`/`score` trong kết quả = thứ hạng theo |w| của SVM fitness train trên tập gene cuối (tập gene là kết quả của wrapper, thứ hạng chỉ để tham khảo).

**Lịch sử (mỗi fold 2 file):** `history/fold_k.csv` — mỗi thế hệ 1 dòng: `best_fitness` (tốt nhất tới thời điểm đó), `iter_best`/`iter_mean`, `diversity` (trung bình khoảng cách Jaccard giữa các cặp cá thể), `exploration`/`exploitation` (tỉ lệ cá thể có khoảng cách Jaccard tới cá thể gần nhất của thế hệ trước > `EXPLORATION_THRESHOLD`), `improvements`, `iter_time_sec`, `elapsed_sec`, `n_evals`; `population_state_history/fold_k.csv` — mỗi cá thể mỗi thế hệ 1 dòng: `generation, individual, fitness, mask` (mask dạng `"[0, 1, ...]"`, đúng định dạng `visualization.ipynb`). Hai file được ghi sau khi đo tài nguyên; phần tính diversity/exploration nằm trong vòng lặp nên có tính vào thời gian.

- **Đầu vào:** `DATA_DIR/fold_{1..5}/train.h5` (chỉ dùng tập **train**; val/test không đụng tới). Dữ liệu giữ nguyên log2 expression, không chuẩn hoá.
- **Đổi dataset:** sửa `DATA_DIR` ở cell cấu hình. Số gene chọn: `N_FEATURES` (mặc định 100).
- **Đầu ra** (`/kaggle/working/feature_selection/ibabc_cgo/<dataset>/`):
  - `selected_genes/fold_k.csv`, `selected_genes_all_folds.csv`, `selected_genes.json` — gene được chọn theo fold (rank, score)
  - `metrics_per_fold.csv`, `metrics_summary.csv` — thời gian, CPU utilization, peak memory, energy, EDP, carbon footprint
- **Đo lường:** chỉ bao quanh lời gọi `select_features` (không tính thời gian đọc file). Energy/Carbon ước lượng bằng `codecarbon` (tracking_mode="process"); EDP = Energy(J) × Time(s).

In [ ]:
# tables: backend cho pd.read_hdf | codecarbon: do nang luong/CO2 | psutil: CPU & RAM
get_ipython().system("pip install -q codecarbon psutil tables")

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
# Doi DATA_DIR sang bo khac (vd .../h5/tcga_luad) de chay dataset khac
DATA_DIR = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5/gse68465"
N_FEATURES = 100          # so gene giu lai moi fold
N_FOLDS = 5
SEED = 42
METHOD = "ibabc_cgo"
# Fitness dung chung: balanced accuracy cua SVC(kernel="linear", class_weight="balanced").
# C = 1e-4 (tune tren TOAN BO ~12.5k gene, gse68465 grid 128/179) qua manh cho tap 100 gene -> fitness phang 0.5.
# "auto": moi fold chon C tren FITNESS_C_GRID theo bacc TB cua FITNESS_CALIB_SUBSETS tap k gene ngau nhien.
# Dat so cu the (vd 1e-4) de dung C co dinh.
FITNESS_SVM_PARAMS = {"C": "auto", "class_weight": "balanced"}
FITNESS_C_GRID = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
FITNESS_CALIB_SUBSETS = 10
FITNESS_HOLDOUT = 0.2          # ti le TRAIN cua fold tach lam holdout de tinh fitness (khong dung val/test)
# Lich su: exploration = ti le ca the cach ca the gan nhat the he truoc > nguong (khoang cach Jaccard)
EXPLORATION_THRESHOLD = 0.1    # 0.1 ~ khac nhau hon ~10 gene tren 100
POPULATION_GZIP = False        # True -> population_state_history/fold_k.csv.gz (nho hon nhieu)
IBABC_BEES = 30
IBABC_ISLANDS = 5
IBABC_ITER = 100
IBABC_MIG_FREQ = 5
IBABC_MIG_RATE = 0.20
IBABC_LIMIT = None             # None = 0.1 * SN * d (Table 3)
IBABC_TRANSFER = "S"          # "S" | "V"
IBABC_RESET_PATIENCE = 10
METHOD_PARAMS = {"bees": IBABC_BEES, "islands": IBABC_ISLANDS, "iter": IBABC_ITER, "mig_freq": IBABC_MIG_FREQ,
                 "mig_rate": IBABC_MIG_RATE, "limit": IBABC_LIMIT, "transfer": IBABC_TRANSFER,
                 "reset_patience": IBABC_RESET_PATIENCE, "fitness": "balanced_accuracy", "fitness_svm": FITNESS_SVM_PARAMS, "fitness_c_grid": FITNESS_C_GRID,
                 "fitness_calib_subsets": FITNESS_CALIB_SUBSETS, "fitness_holdout": FITNESS_HOLDOUT}
# None -> codecarbon tu dinh vi (can Internet); hoac dat ma ISO 3 ky tu, vd "USA", "VNM"
COUNTRY_ISO_CODE = None

DATASET_NAME = os.path.basename(os.path.normpath(DATA_DIR))
RESULT_DIR = os.path.join("/kaggle/working/feature_selection", METHOD, DATASET_NAME)
os.makedirs(os.path.join(RESULT_DIR, "selected_genes"), exist_ok=True)

assert os.path.isfile(os.path.join(DATA_DIR, "fold_1", "train.h5")), \
    f"Khong thay {DATA_DIR}/fold_1/train.h5 -- da Add Input dataset chua?"
print("Dataset :", DATASET_NAME, "->", DATA_DIR)
print("Method  :", METHOD, "| N_FEATURES =", N_FEATURES)
print("Ket qua :", RESULT_DIR)
print("So CPU  :", os.cpu_count())

## Đọc dữ liệu

In [ ]:
import json, time, threading, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}


def load_split(fold, split="train"):
    # File h5 (pandas, key="data"): index = sample_id, cot = ten gene + cot "label"
    df = pd.read_hdf(os.path.join(DATA_DIR, f"fold_{fold}", f"{split}.h5"), key="data")
    X = df.drop(columns="label")
    # Giu nguyen gia tri log2 expression, KHONG chuan hoa
    return X.values.astype(np.float64), df["label"].values.astype(int), X.columns.to_numpy(), df.index.to_numpy()


# Doc thu fold 1 de xem cau truc
X0, y0, genes0, ids0 = load_split(1)
print(f"fold_1/train: {X0.shape[0]} mau x {X0.shape[1]} gene | khoang gia tri [{X0.min():.2f}, {X0.max():.2f}]")
print("Vi du gene:", list(genes0[:5]), "| vi du sample:", list(ids0[:3]))
print("Phan bo lop:", pd.Series(y0).map(STAGE_NAMES).value_counts().to_dict())
del X0, y0, genes0, ids0

## Bộ đo tài nguyên

In [ ]:
import psutil

try:
    from codecarbon import EmissionsTracker, OfflineEmissionsTracker
    _CODECARBON = True
except ImportError:
    _CODECARBON = False


class ResourceTracker:
    """Do tren 1 khoi lenh: thoi gian (s), CPU utilization, peak memory (RSS, gom ca
    tien trinh con cua joblib), energy (kWh, codecarbon), carbon (kg CO2eq) va
    Energy-Delay Product EDP = Energy(J) * Time(s)."""

    def __init__(self, name, interval=0.05):
        self.name, self.interval = name, interval
        self.proc = psutil.Process()
        self._stop = threading.Event()

    def _procs(self):
        try:
            return [self.proc] + self.proc.children(recursive=True)
        except psutil.Error:
            return [self.proc]

    def _poll(self):
        # RSS tong va CPU time cua worker con (tru phan CPU worker da dung truoc khi bat dau do)
        rss = 0
        for p in self._procs():
            try:
                rss += p.memory_info().rss
                if p.pid != self.proc.pid:
                    t = p.cpu_times()
                    self._child_cpu[p.pid] = t.user + t.system - self._child_base.get(p.pid, 0.0)
            except psutil.Error:
                pass
        self._peak_rss = max(self._peak_rss, rss)
        return rss

    def _sampler(self):
        while not self._stop.is_set():
            self._poll()
            self._sys_cpu.append(psutil.cpu_percent(interval=None))
            self._stop.wait(self.interval)

    def __enter__(self):
        self._child_cpu, self._sys_cpu, self._peak_rss = {}, [], 0
        self._child_base = {}
        for p in self._procs()[1:]:
            try:
                t = p.cpu_times()
                self._child_base[p.pid] = t.user + t.system
            except psutil.Error:
                pass
        self.tracker = None
        if _CODECARBON:
            try:
                kw = dict(project_name=self.name, output_dir=RESULT_DIR, save_to_file=False,
                          log_level="error", measure_power_secs=1, tracking_mode="process")
                self.tracker = (OfflineEmissionsTracker(country_iso_code=COUNTRY_ISO_CODE, **kw)
                                if COUNTRY_ISO_CODE else EmissionsTracker(**kw))
                self.tracker.start()
            except Exception as e:
                print("[ResourceTracker] codecarbon loi:", e)
                self.tracker = None
        self.base_rss = self._poll()
        self._child_cpu = {}
        psutil.cpu_percent(interval=None)
        t = self.proc.cpu_times()
        self._cpu0 = t.user + t.system
        self._stop.clear()
        self._thread = threading.Thread(target=self._sampler, daemon=True)
        self._thread.start()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.time_sec = time.perf_counter() - self._t0
        t = self.proc.cpu_times()
        main_cpu = t.user + t.system - self._cpu0
        self._stop.set()
        self._thread.join(timeout=2)
        self._poll()
        cpu_sec = main_cpu + sum(self._child_cpu.values())

        self.energy_kwh, self.carbon_kg = float("nan"), float("nan")
        if self.tracker is not None:
            try:
                self.carbon_kg = float(self.tracker.stop() or 0.0)
                data = self.tracker.final_emissions_data
                self.energy_kwh = float(data.energy_consumed) if data is not None else float("nan")
            except Exception as e:
                print("[ResourceTracker] codecarbon stop loi:", e)

        n_cpu = os.cpu_count() or 1
        self.metrics = {
            "time_sec": self.time_sec,
            "cpu_time_sec": cpu_sec,
            # % tren tong nang luc may (100% = dung het n_cpu core)
            "cpu_util_percent": 100.0 * cpu_sec / (self.time_sec * n_cpu) if self.time_sec > 0 else float("nan"),
            "avg_cores_used": cpu_sec / self.time_sec if self.time_sec > 0 else float("nan"),
            "system_cpu_percent_mean": float(np.mean(self._sys_cpu)) if self._sys_cpu else float("nan"),
            "peak_memory_mb": self._peak_rss / 1024 ** 2,
            "peak_memory_increase_mb": (self._peak_rss - self.base_rss) / 1024 ** 2,
            "energy_kwh": self.energy_kwh,
            "energy_joule": self.energy_kwh * 3.6e6,
            "edp_joule_sec": self.energy_kwh * 3.6e6 * self.time_sec,
            "carbon_kg_co2eq": self.carbon_kg,
            "carbon_g_co2eq": self.carbon_kg * 1e3,
        }
        return False

## Thuật toán: iBABC-CGO (island Binary ABC + Chaos Game Optimization)

In [ ]:
import gzip
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split

SELECT_INFO = {}   # thong tin phu cua lan chon gan nhat -> metrics_per_fold.csv
HISTORY = []       # lich su tung the he cua lan chon gan nhat -> history/fold_k.csv
POP_STATE = []     # (generation, individual, fitness, idx) -> population_state_history/fold_k.csv
_STATE = {}        # bien noi bo: thoi diem bat dau / vong truoc, mask quan the the he truoc


class Fitness:
    """Ham fitness dung chung cho moi wrapper: balanced accuracy cua SVC(kernel="linear") train tren
    phan fit, danh gia tren phan holdout (tach 1 lan, stratified, tu TRAIN cua fold -- khong dung val/test).
    Khong phat theo so gene vi so gene da co dinh = k."""

    def __init__(self, X, y, k):
        fit_i, hold_i = train_test_split(np.arange(len(y)), test_size=FITNESS_HOLDOUT, stratify=y,
                                         random_state=SEED)
        self.Xf, self.yf = np.asfortranarray(X[fit_i]), y[fit_i]     # Fortran order -> lay cot nhanh
        self.Xh, self.yh = np.asfortranarray(X[hold_i]), y[hold_i]
        self.params = dict(FITNESS_SVM_PARAMS)
        if self.params.get("C") == "auto":
            self.params["C"] = self._calibrate_C(X.shape[1], k)
        self.n_evals = 0

    def model(self):
        return SVC(kernel="linear", random_state=SEED, **self.params)

    def _score(self, idx, params):
        clf = SVC(kernel="linear", random_state=SEED, **params).fit(self.Xf[:, idx], self.yf)
        return float(balanced_accuracy_score(self.yh, clf.predict(self.Xh[:, idx])))

    def _calibrate_C(self, p, k):
        # C tune tren toan bo gene KHONG dung lai duoc cho tap k gene (|w.x| co gian theo so gene):
        # chon C tren FITNESS_C_GRID theo balanced accuracy trung binh cua FITNESS_CALIB_SUBSETS tap k gene
        # ngau nhien (chi dung phan fit/holdout cua TRAIN; khong tinh vao n_fitness_evals).
        rng_c = np.random.default_rng(SEED + 1)
        subsets = [random_subset(p, k, rng_c) for _ in range(FITNESS_CALIB_SUBSETS)]
        rows = []
        for C in FITNESS_C_GRID:
            prm = dict(self.params, C=C)
            s = [self._score(ix, prm) for ix in subsets]
            rows.append({"C": C, "bacc_mean": np.mean(s), "bacc_std": np.std(s)})
        tab = pd.DataFrame(rows)
        best = tab.loc[tab.bacc_mean.idxmax()]           # hoa -> C nho nhat (idxmax lay dong dau)
        print("    [fitness] hieu chinh C tren", FITNESS_CALIB_SUBSETS, "tap", k, "gene ngau nhien:")
        print("      " + tab.round(4).to_string(index=False).replace("\n", "\n      "))
        print(f"    [fitness] chon C = {best.C:g} (bacc TB {best.bacc_mean:.4f})")
        SELECT_INFO.update(fitness_C=float(best.C), calib_bacc_mean=float(best.bacc_mean))
        return float(best.C)

    def __call__(self, idx):
        self.n_evals += 1
        return self._score(idx, self.params)

    def coef_norm(self, idx):
        clf = self.model().fit(self.Xf[:, idx], self.yf)
        return np.sqrt((np.atleast_2d(clf.coef_) ** 2).sum(0))


def topk(scores, k, rng=None):
    # Rang buoc |S| = k: lay k chieu co diem lon nhat (hoa -> pha ngau nhien neu co rng)
    if rng is not None:
        scores = scores + rng.random(scores.shape) * 1e-9
    return np.sort(np.argpartition(-scores, k - 1)[:k])


def random_subset(p, k, rng):
    return np.sort(rng.choice(p, size=k, replace=False))


def to_mask(idx, p):
    m = np.zeros(p)
    m[idx] = 1.0
    return m


def _start():
    SELECT_INFO.clear()
    HISTORY.clear()
    POP_STATE.clear()
    _STATE.clear()
    _STATE["t0"] = _STATE["t_last"] = time.perf_counter()
    return np.random.default_rng(SEED)


def _mask_matrix(idxs, p):
    M = np.zeros((len(idxs), p), dtype=np.float32)
    for r, ix in enumerate(idxs):
        M[r, ix] = 1.0
    return M


def _jaccard_dist(A, B):
    # Khoang cach Jaccard giua cac tap gene: 1 - |A n B| / |A u B| (A, B: ma tran mask 0/1)
    inter = A @ B.T
    union = A.sum(1)[:, None] + B.sum(1)[None, :] - inter
    return 1.0 - inter / np.maximum(union, 1.0)


def _log(it, n_iter, best_fit, pop_idxs, pop_fits, fitness, improvements):
    """Ghi 1 the he: HISTORY (tong hop) + POP_STATE (tung ca the).
    - diversity   : trung binh khoang cach Jaccard giua moi cap ca the trong quan the hien tai
    - exploration : ti le ca the co khoang cach Jaccard toi ca the GAN NHAT cua the he truoc > EXPLORATION_THRESHOLD
    - exploitation: 1 - exploration (the he dau tien: NaN)
    - improvements: so ca the tot hon trang thai truoc cua chinh no (dinh nghia cu the xem tung thuat toan)"""
    now = time.perf_counter()
    pop_fits = np.asarray(pop_fits, dtype=float)
    p = fitness.Xf.shape[1]
    M = _mask_matrix(pop_idxs, p)
    D = _jaccard_dist(M, M)
    n = len(pop_idxs)
    diversity = float(D[np.triu_indices(n, 1)].mean()) if n > 1 else 0.0
    prev = _STATE.get("prev_M")
    if prev is None:
        exploration = exploitation = float("nan")
    else:
        exploration = float((_jaccard_dist(M, prev).min(1) > EXPLORATION_THRESHOLD).mean())
        exploitation = 1.0 - exploration
    _STATE["prev_M"] = M
    HISTORY.append({"generation": it, "best_fitness": best_fit, "iter_best": pop_fits.max(),
                    "iter_mean": pop_fits.mean(), "diversity": diversity, "exploration": exploration,
                    "exploitation": exploitation, "improvements": improvements,
                    "iter_time_sec": now - _STATE["t_last"], "elapsed_sec": now - _STATE["t0"],
                    "n_evals": fitness.n_evals})
    POP_STATE.extend((it, i, float(f), np.asarray(ix)) for i, (ix, f) in enumerate(zip(pop_idxs, pop_fits)))
    if it % max(1, n_iter // 10) == 0 or it == n_iter:
        print(f"    gen {it:4d}/{n_iter} | best bacc {best_fit:.4f} | gen best {pop_fits.max():.4f} "
              f"| mean {pop_fits.mean():.4f} | div {diversity:.3f} | explor {exploration:.2f} "
              f"| improv {improvements} | evals {fitness.n_evals}")
    _STATE["t_last"] = time.perf_counter()


def save_population_state(path, p):
    # Cot generation, individual, fitness, mask ("[0, 1, ...]" do dai p) -- dung dinh dang visualization.ipynb
    os.makedirs(os.path.dirname(path), exist_ok=True)
    opener = gzip.open if path.endswith(".gz") else open
    m = np.zeros(p, dtype=np.int8)
    with opener(path, "wt", newline="") as f:
        f.write("generation,individual,fitness,mask\n")
        for gen, ind, fit, idx in POP_STATE:
            m[:] = 0
            m[idx] = 1
            f.write(f'{gen},{ind},{fit:.6f},"[{", ".join(map(str, m.tolist()))}]"\n')
    print(f"  population_state_history: {len(POP_STATE)} dong -> {path} ({os.path.getsize(path) / 1e6:.1f} MB)")


def _finish(fitness, best_idx, best_fit, n_iter):
    # Tap gene la ket qua cua wrapper; thu hang (rank) ben trong tap = |w| cua SVM fitness tren tap do
    idx = np.sort(np.asarray(best_idx))
    w = fitness.coef_norm(idx)
    order = np.argsort(-w, kind="stable")
    SELECT_INFO.update(best_fitness_holdout_bacc=best_fit, n_fitness_evals=fitness.n_evals, n_iterations=n_iter)
    return idx[order], w[order]


def tf_S(x):
    return 1.0 / (1.0 + np.exp(-x))


def tf_V(x):
    return np.abs(np.tanh(x))


def decode(x, tf, k, rng):
    # Ban goc: bit = 1 khi rand < tf(x). Rang buoc k: chon k gene co (tf(x) - rand) lon nhat.
    return topk(tf(x) - rng.random(x.shape), k)


def sample_cgo_alpha(rng):
    # Eq. (7): alpha = Rand | 2 Rand | delta Rand + 1 | eps Rand + (1 - eps)
    c = rng.integers(4)
    if c == 0:
        return rng.random()
    if c == 1:
        return 2.0 * rng.random()
    if c == 2:
        return rng.random() * rng.random() + 1.0
    eps = rng.random()
    return eps * rng.random() + (1.0 - eps)


def cgo_seeds(Xi, MG, GB, rng):
    # Eq. (3)-(6): 3 hat giong tu GB / MG / Xi + 1 hat nhieu 1 chieu ngau nhien
    alpha, beta, gamma = sample_cgo_alpha(rng), float(rng.integers(2)), float(rng.integers(2))
    s4 = Xi.copy()
    s4[rng.integers(len(Xi))] += rng.random()
    return [Xi + alpha * (beta * GB - gamma * MG), GB + alpha * (beta * Xi - gamma * MG),
            MG + alpha * (beta * Xi - gamma * GB), s4]


def select_features(X, y, k):
    """iBABC-CGO (Nssibi et al. 2024), giu logic notebook ibabccgo: dao (island) ABC doc lap
    (employed / onlooker / scout), moi IBABC_MIG_FREQ vong di cu bang 4 hat giong CGO (chon hat tot nhat),
    reset 20% quan the sau 10 vong khong cai thien. Fitness o day la balanced accuracy (CANG LON CANG TOT),
    nen xac suat onlooker / roulette di cu dao chieu so voi ban goc (ban goc cuc tieu)."""
    rng = _start()
    fitness = Fitness(X, y, k)
    dim, SN = X.shape[1], IBABC_BEES
    limit = IBABC_LIMIT if IBABC_LIMIT is not None else int(0.1 * SN * dim)   # Table 3: 0.1 * SN * d
    tf = tf_S if IBABC_TRANSFER == "S" else tf_V

    pop = rng.uniform(-1, 1, size=(SN, dim))
    sel = [decode(pop[i], tf, k, rng) for i in range(SN)]
    fit = np.array([fitness(s) for s in sel])
    trial = np.zeros(SN)
    b = int(np.argmax(fit))
    gbest, gbest_fit, gbest_idx = pop[b].copy(), float(fit[b]), sel[b]
    islands = np.array_split(np.arange(SN), IBABC_ISLANDS)
    no_improve = 0
    _log(0, IBABC_ITER, gbest_fit, sel, fit, fitness, improvements=np.nan)

    def try_replace(i, v):
        idx = decode(v, tf, k, rng)
        f = fitness(idx)
        if f > fit[i]:
            pop[i], fit[i], sel[i], trial[i] = v, f, idx, 0
        else:
            trial[i] += 1

    def partner(isl, li):
        others = [j for j in range(len(isl)) if j != li]
        return isl[rng.choice(others)] if others else isl[li]

    for it in range(IBABC_ITER):
        fit_start = fit.copy()      # improvements = so ong co fitness cuoi vong > dau vong
        for isl in islands:
            for li, i in enumerate(isl):                                   # employed bees
                phi = rng.uniform(-1, 1, size=dim)
                try_replace(i, pop[i] + phi * (pop[i] - pop[partner(isl, li)]))
            f_isl = fit[isl]
            prob = (f_isl - f_isl.min()) / (f_isl.max() - f_isl.min() + 1e-9)
            for li, i in enumerate(isl):                                   # onlooker bees
                if len(isl) > 1 and rng.random() < prob[li]:
                    phi = rng.uniform(-1, 1, size=dim)
                    try_replace(i, pop[i] + phi * (pop[i] - pop[partner(isl, li)]))
            for i in isl:                                                  # scout bees
                if trial[i] > limit:
                    if rng.random() < 0.5:
                        pop[i] = rng.uniform(-1, 1, size=dim)
                    else:
                        m = rng.random(dim) < 0.3
                        pop[i][m] = rng.uniform(-1, 1, size=int(m.sum()))
                    sel[i] = decode(pop[i], tf, k, rng)
                    fit[i], trial[i] = fitness(sel[i]), 0

        if it > 0 and it % IBABC_MIG_FREQ == 0:                            # di cu CGO
            for isl in islands:
                MG = pop[isl].mean(0)
                w = (fit[isl] - fit[isl].min()) + 1e-9
                n_mig = max(1, int(IBABC_MIG_RATE * len(isl)))
                for li in rng.choice(len(isl), size=n_mig, replace=True, p=w / w.sum()):
                    gi = isl[li]
                    best = None
                    for seed in cgo_seeds(pop[gi], MG, gbest, rng):
                        idx = decode(seed, tf, k, rng)
                        f = fitness(idx)
                        if best is None or f > best[1]:
                            best = (seed, f, idx)
                    if best[1] > fit[gi]:
                        pop[gi], fit[gi], sel[gi], trial[gi] = best[0], best[1], best[2], 0

        b = int(np.argmax(fit))
        if fit[b] > gbest_fit:
            gbest, gbest_fit, gbest_idx = pop[b].copy(), float(fit[b]), sel[b]
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= IBABC_RESET_PATIENCE:
            for i in rng.choice(SN, size=int(0.2 * SN), replace=False):
                pop[i] = rng.uniform(-1, 1, size=dim)
                sel[i] = decode(pop[i], tf, k, rng)
                fit[i], trial[i] = fitness(sel[i]), 0
            no_improve = 0
        _log(it + 1, IBABC_ITER, gbest_fit, sel, fit, fitness, improvements=int((fit > fit_start).sum()))

    # Ban goc giai ma lai global_best (ngau nhien) o cuoi; o day tra dung tap gene da duoc danh gia
    return _finish(fitness, gbest_idx, gbest_fit, IBABC_ITER)

## Chạy trên 5 fold

In [ ]:
all_rows, metric_rows = [], []

for fold in range(1, N_FOLDS + 1):
    X, y, genes, _ = load_split(fold, "train")   # CHI dung tap train cho feature selection
    k = min(N_FEATURES, X.shape[1])

    with ResourceTracker(f"{METHOD}_{DATASET_NAME}_fold{fold}") as rt:
        idx, scores = select_features(X, y, k)

    sel = pd.DataFrame({"fold": fold, "rank": np.arange(1, len(idx) + 1),
                        "gene": genes[idx], "score": scores})
    sel.to_csv(os.path.join(RESULT_DIR, "selected_genes", f"fold_{fold}.csv"), index=False)
    # Luu lich su (ngoai phan do tai nguyen): history/fold_k.csv + population_state_history/fold_k.csv
    os.makedirs(os.path.join(RESULT_DIR, "history"), exist_ok=True)
    pd.DataFrame(HISTORY).assign(fold=fold).to_csv(os.path.join(RESULT_DIR, "history", f"fold_{fold}.csv"), index=False)
    save_population_state(os.path.join(RESULT_DIR, "population_state_history",
                                       f"fold_{fold}.csv" + (".gz" if POPULATION_GZIP else "")), X.shape[1])
    all_rows.append(sel)

    m = {"dataset": DATASET_NAME, "method": METHOD, "fold": fold,
         "n_train": X.shape[0], "n_genes_total": X.shape[1], "n_selected": len(idx),
         **SELECT_INFO, **rt.metrics}
    metric_rows.append(m)
    print(f"fold {fold}: {X.shape} -> {len(idx)} gene | {m['time_sec']:.2f}s | CPU {m['cpu_util_percent']:.1f}% "
          f"({m['avg_cores_used']:.2f} core) | peak {m['peak_memory_mb']:.0f} MB | "
          f"{m['energy_kwh']:.3e} kWh | {m['carbon_g_co2eq']:.4f} gCO2eq | top5: {list(sel.gene[:5])}")
    del X, y

selected = pd.concat(all_rows, ignore_index=True)
selected.to_csv(os.path.join(RESULT_DIR, "selected_genes_all_folds.csv"), index=False)
with open(os.path.join(RESULT_DIR, "selected_genes.json"), "w") as f:
    json.dump({f"fold_{r}": g.gene.tolist() for r, g in selected.groupby("fold")}, f, indent=1)

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(os.path.join(RESULT_DIR, "metrics_per_fold.csv"), index=False)
metrics

## Tổng hợp độ đo

In [ ]:
cols = ["time_sec", "cpu_time_sec", "cpu_util_percent", "avg_cores_used", "system_cpu_percent_mean",
        "peak_memory_mb", "peak_memory_increase_mb", "energy_kwh", "energy_joule",
        "edp_joule_sec", "carbon_kg_co2eq", "carbon_g_co2eq"]
summary = metrics[cols].agg(["mean", "std", "min", "max"]).T
summary.loc["time_sec", "total"] = metrics["time_sec"].sum()
summary.loc["energy_kwh", "total"] = metrics["energy_kwh"].sum()
summary.loc["carbon_kg_co2eq", "total"] = metrics["carbon_kg_co2eq"].sum()
summary.to_csv(os.path.join(RESULT_DIR, "metrics_summary.csv"))

with open(os.path.join(RESULT_DIR, "config.json"), "w") as f:
    json.dump({"dataset": DATASET_NAME, "data_dir": DATA_DIR, "method": METHOD, "n_features": N_FEATURES,
               "n_folds": N_FOLDS, "seed": SEED, "params": METHOD_PARAMS, "cpu_count": os.cpu_count(),
               "country_iso_code": COUNTRY_ISO_CODE}, f, indent=1)
summary

## Độ ổn định tập gene giữa các fold (tham khảo)

In [ ]:
# Tham khao nhanh: do on dinh tap gene giua cac fold (Jaccard cap doi, trung binh)
sets = {r: set(g.gene) for r, g in selected.groupby("fold")}
folds = sorted(sets)
jac = pd.DataFrame(index=folds, columns=folds, dtype=float)
for a in folds:
    for b in folds:
        jac.loc[a, b] = len(sets[a] & sets[b]) / len(sets[a] | sets[b])
pairs = [jac.loc[a, b] for i, a in enumerate(folds) for b in folds[i + 1:]]
print(f"Jaccard trung binh giua cac fold: {np.mean(pairs):.3f}")
freq = selected.gene.value_counts()
print(f"Gene xuat hien o ca {N_FOLDS} fold: {int((freq == N_FOLDS).sum())}")
freq.rename("n_folds").to_csv(os.path.join(RESULT_DIR, "gene_frequency.csv"))
jac.round(3)

In [ ]:
import shutil
zip_path = shutil.make_archive(f"/kaggle/working/fs_{METHOD}_{DATASET_NAME}", "zip", RESULT_DIR)
print("Da nen:", zip_path)
for root, _, files in os.walk(RESULT_DIR):
    for fn in sorted(files):
        print(" ", os.path.relpath(os.path.join(root, fn), RESULT_DIR))